# **07_modelO_prophet**

Este notebook implementa el modelo Prophet para la predicción de la demanda eléctrica diaria,
incorporando tendencia, estacionalidad y variables exógenas.

## Objetivos
- Preparar la serie temporal para su uso con Prophet.
- Incorporar festivos y variables regresoras.
- Entrenar y evaluar el modelo sobre el conjunto de prueba.
- Comparar su desempeño con otros enfoques del TFM.
- Generar tablas y figuras útiles para el capítulo de resultados y apéndices.

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

**BLOQUE 3 — Rutas**

In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 4 — Carga de datos**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

DATE_COL = "date"
TARGET_COL = "target_kwh"

df = df.sort_values(DATE_COL).copy()

print("Dimensión del dataset:", df.shape)
display(df.head())
display(df.tail())

**BLOQUE 5 — División train/test**

In [ ]:
train = df[(df[DATE_COL] >= "2022-01-01") & (df[DATE_COL] <= "2024-12-31")].copy()
test = df[(df[DATE_COL] >= "2025-01-01") & (df[DATE_COL] <= "2025-12-10")].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Rango train:", train[DATE_COL].min(), "->", train[DATE_COL].max())
print("Rango test :", test[DATE_COL].min(), "->", test[DATE_COL].max())

**BLOQUE 6 — Funciones auxiliares**

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred)
    }

**BLOQUE 7 — Preparar dataframe Prophet**

In [ ]:
prophet_train = train.rename(columns={DATE_COL: "ds", TARGET_COL: "y"}).copy()
prophet_test = test.rename(columns={DATE_COL: "ds", TARGET_COL: "y"}).copy()

display(prophet_train[["ds", "y"]].head())
display(prophet_test[["ds", "y"]].head())

**BLOQUE 8 — Definir festivo**

In [ ]:
holidays_df = df[df["is_holiday"] == 1][[DATE_COL, "holiday_name"]].copy()
holidays_df = holidays_df.rename(columns={DATE_COL: "ds", "holiday_name": "holiday"})
holidays_df["lower_window"] = 0
holidays_df["upper_window"] = 0

print("Número de registros de festivos:", len(holidays_df))
display(holidays_df.head(10))

**BLOQUE 9 — Definir regresores**

In [ ]:
regressor_cols = [
    "t_mean_c",
    "t_max_c",
    "t_min_c",
    "rh_mean_pct",
    "precip_mm",
    "sun_hours",
    "sst_c",
    "is_weekend",
    "is_business_day",
    "is_holiday",
    "is_pre_holiday",
    "is_post_holiday",
    "is_long_weekend"
]

missing_regressors = [col for col in regressor_cols if col not in prophet_train.columns]
if missing_regressors:
    raise ValueError(f"Faltan regresores: {missing_regressors}")

print("Regresores:", regressor_cols)

**BLOQUE 10 — Construcción final de train/test para Prophet**

In [ ]:
prophet_train_final = prophet_train[["ds", "y"] + regressor_cols].copy()
prophet_test_final = prophet_test[["ds", "y"] + regressor_cols].copy()

print("Train Prophet final:", prophet_train_final.shape)
print("Test Prophet final :", prophet_test_final.shape)

display(prophet_train_final.head())

**BLOQUE 11 — Definir y entrenar el modelo**

In [ ]:
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=holidays_df,
    seasonality_mode="additive",
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0
)

for reg in regressor_cols:
    prophet_model.add_regressor(reg)

prophet_model.fit(prophet_train_final)

**BLOQUE 12 — Predicción en test**

In [ ]:
forecast_test = prophet_model.predict(prophet_test_final.drop(columns=["y"]))

prophet_pred = forecast_test["yhat"].values
y_true = prophet_test_final["y"].values

print("Predicciones Prophet:", prophet_pred.shape)
print("Valores reales      :", y_true.shape)

display(forecast_test[["ds", "yhat", "yhat_lower", "yhat_upper"]].head())

**BLOQUE 13 — Métricas**

In [ ]:
prophet_metrics = compute_metrics(y_true, prophet_pred, "Prophet")
display(pd.DataFrame([prophet_metrics]))

**BLOQUE 14 — Gráfico real vs predicción**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(prophet_test_final["ds"], y_true, label="Real", linewidth=1.2)
ax.plot(prophet_test_final["ds"], prophet_pred, label="Predicción Prophet", linewidth=1.2)

ax.set_title("Predicción en test - Prophet", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_prophet.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 15 — Componentes del modelo**

In [ ]:
fig_components = prophet_model.plot_components(forecast_test)
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "prophet_components.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
forecast_test.columns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Tendencia
axes[0, 0].plot(forecast_test["ds"], forecast_test["trend"])
axes[0, 0].set_title("Tendencia")
axes[0, 0].set_xlabel("Fecha")
axes[0, 0].set_ylabel("kWh")

# 2. Estacionalidad semanal
axes[0, 1].plot(forecast_test["ds"], forecast_test["weekly"])
axes[0, 1].set_title("Estacionalidad semanal")
axes[0, 1].set_xlabel("Fecha")

# 3. Estacionalidad anual
axes[1, 0].plot(forecast_test["ds"], forecast_test["yearly"])
axes[1, 0].set_title("Estacionalidad anual")
axes[1, 0].set_xlabel("Fecha")

# 4. Efecto de variables externas (extra regressors)
axes[1, 1].plot(forecast_test["ds"], forecast_test["extra_regressors_additive"])
axes[1, 1].set_title("Efecto de variables externas")
axes[1, 1].set_xlabel("Fecha")

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "prophet_components_2x2.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 16 — Exportación de predicciones**

In [ ]:
prophet_predictions_df = pd.DataFrame({
    "date": prophet_test_final["ds"].values,
    "y_true": y_true,
    "y_pred_prophet": prophet_pred
})

display(prophet_predictions_df.head())
prophet_predictions_df.to_csv(OUTPUTS_TABLES / "table_prophet_test_predictions.csv", index=False)

**BLOQUE 17 — Tabla de hiperparámetros**

In [ ]:
prophet_hyperparams = pd.DataFrame([{
    "model": "Prophet",
    "yearly_seasonality": True,
    "weekly_seasonality": True,
    "daily_seasonality": False,
    "seasonality_mode": "additive",
    "changepoint_prior_scale": 0.05,
    "seasonality_prior_scale": 10.0,
    "holidays_included": True,
    "regressors": ", ".join(regressor_cols)
}])

display(prophet_hyperparams)
prophet_hyperparams.to_csv(OUTPUTS_TABLES / "table_hyperparameters_prophet.csv", index=False)